# Pattern 1: Service Credential

The simplest pattern. The MCP server attaches a shared API key to every outbound request. The backend service validates the key but has no idea which user made the request.

**What the service sees:** 'the agent called me' - it cannot distinguish users.

**Weakness:** No user-level filtering, no audit trail. Every user gets the same data.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p01_service_credential")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭───────────────── p01_service_credential/mcp_auth.py ──────────────────╮
│    1 """Pattern 1: Service Credential.                                │
│    2                                                                  │
│    3 The MCP server uses a single static API key for every tool call. │
│    4 No user identity is forwarded to the backend service.            │
│    5 """                                                              │
│    6                                                                  │
│    7 from framework.config import SHARED_SERVICE_API_KEY              │
│    8 from framework.mcp.auth import AuthHandler                       │
│    9                                                                  │
│   10                                                                  │
│   11 class ServiceCredentialHandler(AuthHandler):                     │
│   12     async def prepare_request(self, user_context, headers):      │
│   13         headers["X-API-Key"] = SHARED_SERVICE_API_KEY            │
│   14         return headers                                           │
│   15                                                                  │
│   16                                                                  │
│   17 auth_handler = ServiceCredentialHandler()                        │
│   18                                                                  │
╰───────────────────────────────────────────────────────────────────────╯

╭────────────────────── p01_service_credential/service_auth.py ───────────────────────╮
│    1 """Pattern 1: Service Credential (service side).                               │
│    2                                                                                │
│    3 The service checks the shared API key. No user identity is extracted.          │
│    4 """                                                                            │
│    5                                                                                │
│    6 from fastapi import Request                                                    │
│    7 from framework.services.identity import Identity                               │
│    8                                                                                │
│    9 EXPECTED_API_KEY = "dev-shared-api-key"                                        │
│   10                                                                                │
│   11                                                                                │
│   12 async def get_expense_identity(request: Request) -> Identity:                  │
│   13     api_key = request.headers.get("x-api-key")                                 │
│   14     if not api_key:                                                            │
│   15         return Identity(method="none", detail="no auth provided")              │
│   16     if api_key == EXPECTED_API_KEY:                                            │
│   17         return Identity(                                                       │
│   18             method="api_key",                                                  │
│   19             detail="shared service credential, no user identity",              │
│   20         )                                                                      │
│   21     return Identity(                                                           │
│   22         method="none",                                                         │
│   23         detail=f"X-API-Key did not match (received prefix: {api_key[:8]}...)", │
│   24     )                                                                          │
│   25                                                                                │
│   26                                                                                │
│   27 # Document service uses the same auth                                          │
│   28 get_document_identity = get_expense_identity                                   │
│   29                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────╯

The MCP server adds X-API-Key to every request. The service checks it matches.
No user context flows through at all.

In [3]:
await runner.start()

[04/14/26 06:16:47] INFO     StreamableHTTP session manager started                  ]8;id=463237;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=691576;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=98072;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=764087;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64782/mcp "HTTP/1.1 200 OK"        ]8;id=134970;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=614859;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=808821;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=972176;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=57227;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=848869;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:64782/mcp "HTTP/1.1 200 OK"        ]8;id=505775;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=587077;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64782/mcp "HTTP/1.1 200 OK"        ]8;id=257450;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=997161;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64783/mcp "HTTP/1.1 200 OK"        ]8;id=276044;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=69004;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=153097;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=55133;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=549897;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=707084;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users. Since the service has no user identity, all three users should see the same data. Any variation in results is the LLM agent's interpretation of the query, not auth-based filtering.

**Note:** the service returns ALL data for every request in this pattern. If the agent appears to filter results (e.g. showing fewer documents for one user), that's the LLM making assumptions from the query wording, not the service enforcing access control.

In [4]:
await runner.run_as("alice", "What are my recent expenses?")

                    INFO     HTTP Request: POST                                                     ]8;id=418170;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=425869;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────╮
│ [alice] What are my recent expenses? │
╰──────────────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=66190;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=874990;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=355384;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=11185;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=456628;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=709052;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=958207;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=223785;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:17:03] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=179726;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=619809;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",                   │
│     │              │      │        │ "identity_detail": "shared service credential, no user identity", "count": │
│     │              │      │        │ 8, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", │
│     │              │      │        │ "amount": 42.5, "category": "software", "description": "JetBrains AI       │
│     │              │      │        │ assistant subscription",                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the recent expenses on your account (based on the current session):                                    │
│                                                                                                                 │
│ - Expense ID 1 — Alice, Engineering — $42.50 — Category: Software — JetBrains AI assistant subscription —       │
│ Status: Approved                                                                                                │
│ - Expense ID 2 — Alice, Engineering — $156.00 — Category: Travel — Train ticket to client offsite — Status:     │
│ Approved                                                                                                        │
│ - Expense ID 3 — Alice, Engineering — $89.00 — Category: Books — Designing Data-Intensive Applications —        │
│ Status: Approved                                                                                                │
│ - Expense ID 4 — Alice, Engineering — $1,450.00 — Category: Hardware — External 4K monitor — Status: Pending    │
│ - Expense ID 5 — Bob, Engineering — $320.00 — Category: Training — OAuth 2.0 deep-dive workshop — Status:       │
│ Approved                                                                                                        │
│ - Expense ID 6 — Bob, Engineering — $67.00 — Category: Meals — Team lunch after the migration shipped — Status: │
│ Approved                                                                                                        │
│ - Expense ID 7 — Dave, Platform — $980.00 — Category: Training — KubeCon ticket — Status: Approved              │
│ - Expense ID 8 — Dave, Platform — $240.00 — Category: Software — Datadog license seat — Status: Approved        │
│                                                                                                                 │
│ If you want only your own expenses, I may need to re-run with your user identity. Tell me your username or      │
│ re-authenticate so I can filter to you. I can also group by status or show totals if you’d like.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are the recent expenses on your account (based on the current session):\n\n- Expense ID 1 — Alice, Engineering — $42.50 — Category: Software — JetBrains AI assistant subscription — Status: Approved\n- Expense ID 2 — Alice, Engineering — $156.00 — Category: Travel — Train ticket to client offsite — Status: Approved\n- Expense ID 3 — Alice, Engineering — $89.00 — Category: Books — Designing Data-Intensive Applications — Status: Approved\n- Expense ID 4 — Alice, Engineering — $1,450.00 — Category: Hardware — External 4K monitor — Status: Pending\n- Expense ID 5 — Bob, Engineering — $320.00 — Category: Training — OAuth 2.0 deep-dive workshop — Status: Approved\n- Expense ID 6 — Bob, Engineering — $67.00 — Category: Meals — Team lunch after the migration shipped — Status: Approved\n- Expense ID 7 — Dave, Platform — $980.00 — Category: Training — KubeCon ticket — Status: Approved\n- Expense ID 8 — Dave, Platform — $240.00 — Category: Software — Datadog license seat 

In [5]:
await runner.run_as("bob", "Show me all expenses and approve expense 3")

                    INFO     HTTP Request: POST                                                     ]8;id=541978;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=392387;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────────────╮
│ [bob] Show me all expenses and approve expense 3 │
╰──────────────────────────────────────────────────╯

[04/14/26 06:17:08] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=138299;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=340425;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:17:09] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=217383;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=772976;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=249468;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=589097;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:17:14] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=576214;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=994028;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:17:24] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=163321;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=184625;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args                 ┃ status ┃ result                                                  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses    │ {'department': None} │      - │ {'type': 'text', 'text': '{"identity_method":           │
│     │                 │                      │        │ "api_key", "identity_detail": "shared service           │
│     │                 │                      │        │ credential, no user identity", "count": 8, "expenses":  │
│     │                 │                      │        │ [{"id": 1, "user_id": "alice", "department":            │
│     │                 │                      │        │ "engineering", "amount": 42.5, "category": "software",  │
│     │                 │                      │        │ "description": "JetBrains AI assistant subscription",   │
│ 2   │ approve_expense │ {'expense_id': 3}    │      - │ {'type': 'text', 'text': '{"identity_method":           │
│     │                 │                      │        │ "api_key", "approved_by": "<api-key-caller>",           │
│     │                 │                      │        │ "tool_side_authz_enabled": false, "expense": {"id": 3,  │
│     │                 │                      │        │ "user_id": "alice", "department": "engineering",        │
│     │                 │                      │        │ "amount": 89.0, "category": "books", "description":     │
│     │                 │                      │        │ "Designing Data-Intensive Applications", "status": "    │
└─────┴─────────────────┴──────────────────────┴────────┴─────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are your expenses:                                                                                         │
│                                                                                                                 │
│ - ID 1 | user: alice | dept: engineering | amount: 42.50 | category: software | description: JetBrains AI       │
│ assistant subscription | status: approved                                                                       │
│ - ID 2 | user: alice | dept: engineering | amount: 156.00 | category: travel | description: Train ticket to     │
│ client offsite | status: approved                                                                               │
│ - ID 3 | user: alice | dept: engineering | amount: 89.00 | category: books | description: Designing             │
│ Data-Intensive Applications | status: approved                                                                  │
│ - ID 4 | user: alice | dept: engineering | amount: 1450.00 | category: hardware | description: External 4K      │
│ monitor | status: pending                                                                                       │
│ - ID 5 | user: bob | dept: engineering | amount: 320.00 | category: training | description: OAuth 2.0 deep-dive │
│ workshop | status: approved                                                                                     │
│ - ID 6 | user: bob | dept: engineering | amount: 67.00 | category: meals | description: Team lunch after the    │
│ migration shipped | status: approved                                                                            │
│ - ID 7 | user: dave | dept: platform | amount: 980.00 | category: training | description: KubeCon ticket |      │
│ status: approved                                                                                                │
│ - ID 8 | user: dave | dept: platform | amount: 240.00 | category: software | description: Datadog license seat  │
│ | status: approved                                                                                              │
│                                                                                                                 │
│ Expense 3 update:                                                                                               │
│ - Expense 3 has been approved (current status: approved). Details: id 3, user alice, dept engineering, amount   │
│ 89.00, category books, description Designing Data-Intensive Applications. Approved by api-key-caller.           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your expenses:\n\n- ID 1 | user: alice | dept: engineering | amount: 42.50 | category: software | description: JetBrains AI assistant subscription | status: approved\n- ID 2 | user: alice | dept: engineering | amount: 156.00 | category: travel | description: Train ticket to client offsite | status: approved\n- ID 3 | user: alice | dept: engineering | amount: 89.00 | category: books | description: Designing Data-Intensive Applications | status: approved\n- ID 4 | user: alice | dept: engineering | amount: 1450.00 | category: hardware | description: External 4K monitor | status: pending\n- ID 5 | user: bob | dept: engineering | amount: 320.00 | category: training | description: OAuth 2.0 deep-dive workshop | status: approved\n- ID 6 | user: bob | dept: engineering | amount: 67.00 | category: meals | description: Team lunch after the migration shipped | status: approved\n- ID 7 | user: dave | dept: platform | amount: 980.00 | category: training | description: 

In [6]:
await runner.run_as("dave", "Search for architecture documents")

[04/14/26 06:17:25] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=92397;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=161983;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST                                                     ]8;id=116537;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=818099;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────╮
│ [dave] Search for architecture documents │
╰──────────────────────────────────────────╯

[04/14/26 06:17:30] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=87527;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=137060;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:17:31] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=248683;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=862415;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

...

[04/14/26 06:17:35] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=214437;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=870533;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:17:37] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=409496;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=677753;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args                  ┃ status ┃ result                                                ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': 'architecture'} │      - │ {'type': 'text', 'text': '{"identity_method":         │
│     │                  │                       │        │ "api_key", "identity_detail": "shared service         │
│     │                  │                       │        │ credential, no user identity", "allowed_groups":      │
│     │                  │                       │        │ "all", "count": 1, "documents": [{"id": 1, "title":   │
│     │                  │                       │        │ "Architecture decision records, Q1", "body":          │
│     │                  │                       │        │ "ADR-014: moved authentication to Keycloak. ADR-015:  │
│     │                  │                       │        │ adopted O                                             │
└─────┴──────────────────┴───────────────────────┴────────┴───────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are the architecture-related documents I found:                                                            │
│                                                                                                                 │
│ - ID 1: Architecture decision records, Q1                                                                       │
│   - Owner: bob                                                                                                  │
│   - Access groups: engineering                                                                                  │
│   - Snippet: ADR-014: moved authentication to Keycloak. ADR-015: adopted OPA for policy.                        │
│                                                                                                                 │
│ Would you like to view the full document (ID 1) or run a more specific search (e.g., system architecture,       │
│ component diagrams)?                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are the architecture-related documents I found:\n\n- ID 1: Architecture decision records, Q1\n  - Owner: bob\n  - Access groups: engineering\n  - Snippet: ADR-014: moved authentication to Keycloak. ADR-015: adopted OPA for policy.\n\nWould you like to view the full document (ID 1) or run a more specific search (e.g., system architecture, component diagrams)?', tool_calls=[ToolCallTrace(name='search_documents', args={'q': 'architecture'}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "api_key", "identity_detail": "shared service credential, no user identity", "allowed_groups": "all", "count": 1, "documents": [{"id": 1, "title": "Architecture decision records, Q1", "body": "ADR-014: moved authentication to Keycloak. ADR-015: adopted O', error=None)])

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [7]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:64780/debug/last-request "HTTP/1.1  ]8;id=231985;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=209478;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── expense-service /debug/last-request ─────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:64781/debug/last-request "HTTP/1.1  ]8;id=673679;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=2032;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── document-service /debug/last-request ────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

Notice the service reports method=api_key for all three users. It has no idea who called. Alice, bob, and dave all see the same data. The service cannot filter by user.

This is the baseline. Every pattern that follows adds something to close this gap.

In [8]:
await runner.stop()

Pattern p01_service_credential stopped.

[04/14/26 06:17:41] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=341428;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=230948;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       